# Disaster assistant with Unsloth (Colab): faster SFT on the mixed data

Same data and same goal as `disaster_assistant_colab.ipynb`, but training runs through [Unsloth](https://unsloth.ai) (faster, less memory).
What is **different** from the plain notebook (so results are not a pure like-for-like): Unsloth trainer and 8-bit optimizer, `lora_dropout = 0` (Unsloth's fast path), and 16-bit LoRA on the `unsloth/Llama-3.2-*-Instruct` copy of the model (same weights, no Hugging Face login needed).
What is **the same**: LoRA r=8 alpha=16 on all 7 layer types, learning rate 2e-4 cosine, effective batch 16, loss on the answer only, the exact same prompt text as the iOS app.

**How to run**: upload `train.jsonl` and `test_eval.jsonl` from `data/splits_multi/`. Run Setup, Config, Helpers. Keep `SMOKE = True` first (5 steps) and check the speed line, then set `SMOKE = False`, re-run Config, and run Part B.
The speed line prints seconds per step. The plain notebook needed about 6 s/step on 3B and 2.4 s/step on 1B (T4), so you can see the gain directly.

## 0. Setup (once per session)

In [ ]:
# If this cell fails, copy the install cell from the official Unsloth Colab notebook for Llama 3.2
# (https://unsloth.ai/docs/get-started/install/google-colab). Versions move quickly.
!pip install -q unsloth

In [ ]:
from google.colab import drive
drive.mount("/content/drive")
!nvidia-smi

## 1. Config (the only cell you edit)

In [ ]:
import os, json, gc, math, time

SIZE  = "3b"     # "1b" or "3b"
SMOKE = True     # True = 5-step test run.  False = the real run.   <-- CHANGE THIS, then re-run this cell

MODEL = {"1b": "unsloth/Llama-3.2-1B-Instruct", "3b": "unsloth/Llama-3.2-3B-Instruct"}[SIZE]
BATCH = {"1b": (8, 2), "3b": (4, 4)}[SIZE]      # (per-device batch, gradient accumulation). Effective batch = 16
STEPS = 5 if SMOKE else -1
TAG   = f"{SIZE}_smoke" if SMOKE else SIZE
EPOCHS = 2

TRAIN_FILE = "train.jsonl"        # one shuffled file, all disasters mixed
TEST_FILE  = "test_eval.jsonl"    # held-out questions, never trained on
RUN_BASE_TEST = True              # also answer the test questions with the untrained model (the "before" numbers)

OUT = f"/content/drive/MyDrive/disaster_unsloth_{SIZE}"
os.makedirs(OUT, exist_ok=True)
ADAPTER_DIR = f"{OUT}/sft_adapter_{TAG}"
MERGED_DIR  = f"{OUT}/merged_16bit_{TAG}"

print(f"model: {MODEL}\nsmoke test: {SMOKE} (steps = {STEPS})\nsaving to: {OUT}")
if SMOKE: print("\n*** SMOKE TEST: results from this run are NOT trained. Set SMOKE = False for real results. ***")
for f in (TRAIN_FILE, TEST_FILE): print(("ok      " if os.path.exists(f) else "MISSING ") + f)

## 2. Helpers (run once, no edits)

In [ ]:
from unsloth import FastLanguageModel      # keep this import BEFORE trl / transformers / peft
import torch
from datasets import Dataset

# The exact prompt the iOS app sends (and the Llama 3.2 template renders), written out by hand.
# Writing it ourselves avoids differences in the date header between chat-template versions.
def render(system, user, assistant=None):
    text = ("<|begin_of_text|><|start_header_id|>system<|end_header_id|>\n\n"
            "Cutting Knowledge Date: December 2023\nToday Date: 26 Jul 2024\n\n" + system + "<|eot_id|>"
            "<|start_header_id|>user<|end_header_id|>\n\n" + user + "<|eot_id|>"
            "<|start_header_id|>assistant<|end_header_id|>\n\n")
    return text if assistant is None else text + assistant + "<|eot_id|>"

def load_rows(path):
    return [json.loads(l) for l in open(path)]

TEST_ROWS = load_rows(TEST_FILE)

def free_gpu(*names):
    for n in names: globals().pop(n, None)
    gc.collect(); torch.cuda.empty_cache()

def run_test(model, tokenizer, name, bs=8, max_new=512):
    """Answer every test question (greedy) and save <name>.json to Drive."""
    FastLanguageModel.for_inference(model)
    tokenizer.padding_side = "left"
    results = []
    for i in range(0, len(TEST_ROWS), bs):
        batch = TEST_ROWS[i:i + bs]
        prompts = [render(r["messages"][0]["content"], r["messages"][1]["content"]) for r in batch]
        enc = tokenizer(prompts, return_tensors="pt", padding=True, add_special_tokens=False).to("cuda")
        with torch.no_grad():
            gen = model.generate(**enc, max_new_tokens=max_new, do_sample=False, use_cache=True)
        for r, seq in zip(batch, gen):
            results.append({"question": r["messages"][1]["content"], "urgency": r["labels"]["urgency"],
                            "disaster": r["labels"].get("disaster"),
                            "answer": tokenizer.decode(seq[enc["input_ids"].shape[1]:], skip_special_tokens=True)})
    path = f"{OUT}/test_{name}.json"
    json.dump(results, open(path, "w"))
    w = [len(x["answer"].split()) for x in results]
    print(f"saved {path} | {len(results)} answers | avg {sum(w)/len(w):.0f} words | mention 911: {sum('911' in x['answer'] for x in results)}")
    return results

def ask(model, tokenizer, question):
    FastLanguageModel.for_inference(model)
    enc = tokenizer(render(TEST_ROWS[0]["messages"][0]["content"], question), return_tensors="pt", add_special_tokens=False).to("cuda")
    out = model.generate(**enc, max_new_tokens=350, do_sample=False, use_cache=True)
    print(tokenizer.decode(out[0][enc["input_ids"].shape[1]:], skip_special_tokens=True))

## Part A. Base model (the 'before' numbers)

In [ ]:
if RUN_BASE_TEST:
    free_gpu("model", "tokenizer", "trainer")
    model, tokenizer = FastLanguageModel.from_pretrained(MODEL, max_seq_length=2048, dtype=None, load_in_4bit=False)
    ask(model, tokenizer, "water up to my knees in basement, stairs half under, what do i do")
    res_base = run_test(model, tokenizer, f"base-{SIZE}")

## Part B. SFT with Unsloth
Full run = 2 epochs over about 5,470 rows, about 680 steps.

In [ ]:
# data: the chat rows become one text each, with the exact prompt text. 95% train / 5% validation.
rows = load_rows(TRAIN_FILE)
texts = [render(r["messages"][0]["content"], r["messages"][1]["content"], r["messages"][2]["content"]) for r in rows]
ds = Dataset.from_dict({"text": texts}).train_test_split(test_size=0.05, seed=42)
sft_train, sft_val = ds["train"], ds["test"]
print(len(sft_train), "train /", len(sft_val), "validation")
print(sft_train[0]["text"][-300:])

In [ ]:
free_gpu("model", "tokenizer", "trainer")
model, tokenizer = FastLanguageModel.from_pretrained(MODEL, max_seq_length=2048, dtype=None, load_in_4bit=False)
model = FastLanguageModel.get_peft_model(
    model, r=8, lora_alpha=16, lora_dropout=0, bias="none",
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
    use_gradient_checkpointing="unsloth", random_state=3407,
)
model.print_trainable_parameters()

In [ ]:
from trl import SFTConfig, SFTTrainer
from unsloth import is_bfloat16_supported

bs, accum = BATCH
cfg = dict(
    output_dir="outputs", dataset_text_field="text", dataset_num_proc=2,
    num_train_epochs=EPOCHS, max_steps=STEPS,
    per_device_train_batch_size=bs, gradient_accumulation_steps=accum,
    learning_rate=2e-4, lr_scheduler_type="cosine", warmup_steps=5, weight_decay=0.01,
    optim="adamw_8bit", seed=3407, logging_steps=5,
    fp16=not is_bfloat16_supported(), bf16=is_bfloat16_supported(),
    eval_strategy="steps" if STEPS > 0 else "epoch", eval_steps=5, save_strategy="no", report_to="none",
)
# trl renamed a few arguments between versions: try the new names first, then the old ones.
try:
    args = SFTConfig(**cfg, max_length=2048)
except TypeError:
    args = SFTConfig(**cfg, max_seq_length=2048)
try:
    trainer = SFTTrainer(model=model, processing_class=tokenizer, train_dataset=sft_train, eval_dataset=sft_val, args=args)
except TypeError:
    trainer = SFTTrainer(model=model, tokenizer=tokenizer, train_dataset=sft_train, eval_dataset=sft_val, args=args)

# Loss on the answer only (not on the system prompt or the question).
from unsloth.chat_templates import train_on_responses_only
trainer = train_on_responses_only(
    trainer,
    instruction_part="<|start_header_id|>user<|end_header_id|>\n\n",
    response_part="<|start_header_id|>assistant<|end_header_id|>\n\n",
)

# Checks: no double start token, and only the answer is trained on.
ex = trainer.train_dataset[0]
ids = ex["input_ids"]
assert not (ids[0] == ids[1] == tokenizer.bos_token_id), "double BOS token: remove <|begin_of_text|> from render() or the tokenizer adds a second one"
trained = tokenizer.decode([t for t, l in zip(ids, ex["labels"]) if l != -100])
print("TEXT THE LOSS IS COMPUTED ON:\n", trained[:300])
assert "Cutting Knowledge" not in trained and "emergency assistant" not in trained, "the system prompt is being trained on"

In [ ]:
t0 = time.time()
trainer.train()
n = trainer.state.global_step
print(f"\n{n} steps in {time.time()-t0:.0f} s  =  {(time.time()-t0)/max(n,1):.1f} s per step   (plain notebook on a T4: about 6 s/step for 3B, 2.4 s/step for 1B)")

In [ ]:
# save the adapter (small) and the merged 16-bit model (needed for GGUF conversion on the Mac)
expected = math.ceil(len(sft_train) / 16) * EPOCHS
steps_done = trainer.state.global_step
model.save_pretrained(ADAPTER_DIR); tokenizer.save_pretrained(ADAPTER_DIR)
print(f"saved adapter {ADAPTER_DIR}\ntrained steps: {steps_done}   (a full run is about {expected})")
if SMOKE or steps_done < 0.9 * expected:
    print("\n*** This model is NOT fully trained. Do not use it for results. ***")
else:
    model.save_pretrained_merged(MERGED_DIR, tokenizer, save_method="merged_16bit")
    print("saved merged model", MERGED_DIR)

In [ ]:
# loss curve
import matplotlib.pyplot as plt
log = trainer.state.log_history
def series(key):
    pts = [(r["step"], r[key]) for r in log if key in r]
    return list(zip(*pts)) if pts else ([], [])
plt.figure(figsize=(7, 4))
plt.plot(*series("loss"), label="train"); plt.plot(*series("eval_loss"), "o-", label="validation")
plt.xlabel("step"); plt.ylabel("loss"); plt.title(f"SFT loss ({TAG})"); plt.legend(); plt.grid(alpha=0.3)
plt.savefig(f"{OUT}/sft_loss_{TAG}.png", dpi=150, bbox_inches="tight"); plt.show()

In [ ]:
# test the trained model on the held-out questions
ask(model, tokenizer, "water up to my knees in basement, stairs half under, what do i do")
res_sft = run_test(model, tokenizer, f"sft-{TAG}")

## 3. Results and files

In [ ]:
import glob
for p in sorted(glob.glob(f"{OUT}/test_*.json")):
    r = json.load(open(p)); w = [len(x["answer"].split()) for x in r]
    print(f"{os.path.basename(p):34} {len(r)} answers | avg {sum(w)/len(w):4.0f} words | 911: {sum('911' in x['answer'] for x in r)}/{len(r)}")
print("\nfiles on Drive:"); print("\n".join(sorted(os.listdir(OUT))))